# Wi-Fi Fingerprint Indoor Localization — Regression Track

**Objective:** Predict the indoor (X, Y) coordinates of a device using Wi-Fi RSSI fingerprints.

**Dataset:** UJIndoorLoc (UCI ML Repository) — 520 WAP signal readings from 3 buildings across multiple floors.

**Team:** K Ganesh Giridhar (519) · G R Balaji (510) · A Suhas Reddy (503)

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded successfully")

## 2. Dataset Loading & Audit

Load the raw UJIndoorLoc training and validation sets. We keep the originals untouched in `data/raw/`.

In [ ]:
train_df = pd.read_csv('../data/raw/trainingData.csv')
val_df   = pd.read_csv('../data/raw/validationData.csv')

print(f"Training set  : {train_df.shape[0]} rows, {train_df.shape[1]} columns")
print(f"Validation set: {val_df.shape[0]} rows, {val_df.shape[1]} columns")

**Observation:** Training set has ~19,937 samples and 529 columns (520 WAPs + 9 metadata). Validation has 1,111 samples.

In [ ]:
# Column types overview
train_df.info(verbose=False)

In [ ]:
# First few rows — WAP columns and target columns
print("WAP columns sample:")
print(train_df.iloc[:3, :5])
print()
print("Target / metadata columns:")
print(train_df[['LONGITUDE','LATITUDE','FLOOR','BUILDINGID','SPACEID']].head())

In [ ]:
# Check for missing values
missing = train_df.isnull().sum().sum()
print(f"Total missing values in training set: {missing}")

**Inference:** No traditional missing values (NaN) exist. However, the RSSI value `+100` is a **sentinel** meaning 'AP not detected' — this needs special treatment later.

In [ ]:
# WAP columns
wap_cols = [c for c in train_df.columns if c.startswith('WAP')]
print(f"Number of WAP features: {len(wap_cols)}")

# Check how many +100 values exist
sentinel_count = (train_df[wap_cols] == 100).sum().sum()
total_cells = train_df[wap_cols].shape[0] * train_df[wap_cols].shape[1]
pct = (sentinel_count / total_cells) * 100
print(f"Sentinel (+100) values: {sentinel_count:,} out of {total_cells:,} ({pct:.1f}%)")

**Key Finding:** A very large proportion of WAP readings are +100 (not detected). This is expected — a device can only see a small subset of all 520 access points from any location.

In [ ]:
# Target variable distributions
print("=== LONGITUDE ===")
print(train_df['LONGITUDE'].describe())
print()
print("=== LATITUDE ===")
print(train_df['LATITUDE'].describe())
print()
print("=== FLOOR distribution ===")
print(train_df['FLOOR'].value_counts().sort_index())
print()
print("=== BUILDING distribution ===")
print(train_df['BUILDINGID'].value_counts().sort_index())

**Observation:**
- Coordinates span a wide range (campus-level) across 3 buildings
- Floor values range from 0 to 4 (5 floors total)
- Building 0, 1, 2 have varying sample counts — slightly imbalanced but acceptable